In [1]:
import pandas as pd

In [2]:
_df = pd.read_csv("inpatient_admissions.csv", parse_dates=["admission_date", "discharge_date"])

In [3]:
_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   admission_id    623 non-null    str           
 1   patient_id      623 non-null    str           
 2   admission_date  623 non-null    datetime64[us]
 3   discharge_date  623 non-null    datetime64[us]
dtypes: datetime64[us](2), str(2)
memory usage: 19.6 KB


In [13]:
df = _df.sort_values(
    ["patient_id", "admission_date"]
).copy()

df = df.assign(
    next_admission_date = lambda x: x.groupby("patient_id")["admission_date"].shift(-1),
    days_to_readmission = lambda x: (x["next_admission_date"] - x["discharge_date"]).dt.days,
    readmission_30d = lambda x: x["days_to_readmission"].between(0, 30, inclusive="both")
)

df

,admission_id,patient_id,admission_date,discharge_date,next_admission_date,days_to_readmission,readmission_30d
83,ADM00084,P0001,2016-01-19,2016-01-20,2018-06-19,881.0,False
255,ADM00256,P0001,2018-06-19,2018-06-20,NaT,NaN,False
108,ADM00109,P0002,2016-05-27,2016-05-28,2017-08-04,433.0,False
188,ADM00189,P0002,2017-08-04,2017-08-05,2018-01-19,167.0,False
222,ADM00223,P0002,2018-01-19,2018-01-20,2024-07-01,2354.0,False
...,...,...,...,...,...,...,...
565,ADM00566,P0127,2023-04-26,2023-04-27,2024-04-26,365.0,False
580,ADM00581,P0127,2024-04-26,2024-04-27,2025-05-10,378.0,False
614,ADM00615,P0127,2025-05-10,2025-05-11,2025-11-06,179.0,False
620,ADM00621,P0127,2025-11-06,2025-11-07,NaT,NaN,False


In [14]:
df["readmission_30d"].sum() / len(df)

np.float64(0.3723916532905297)